# RealMeta — Run the Real Backend on Colab's Free GPU

This notebook runs the **actual RealMeta backend** (the same `api.py` /
`process_video.py` from the `/backend` folder) on a free Google Colab
GPU, and gives it a temporary public web address. Paste that address
into your locally-running website (`localhost:5173`), and its
"Upload Video" button will genuinely process videos on this GPU —
no need to rent or manage your own server for a demo.

**Before you start, get a free ngrok account (one-time, ~1 minute):**
1. Go to [ngrok.com](https://dashboard.ngrok.com/signup) and sign up (free)
2. Go to [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Copy your authtoken — you'll paste it into Step 2 below

**How to use this notebook:**
1. Click **Runtime → Change runtime type**, select **T4 GPU**, click Save
2. Run each cell top to bottom
3. The last cell prints a public URL like `https://abcd-1234.ngrok-free.app`
4. Paste that URL into the "Backend URL" field on your website's upload screen
5. Upload a video on your website as normal — it'll process here, on this GPU

**Important:** this server only exists while this notebook is running.
Closing the tab, letting the session time out, or clicking Stop will
shut it down — that's expected for a free demo setup, not a bug. Each
time you restart this notebook, you'll get a **new** URL to paste in.

## Step 1 — Confirm we have a GPU

In [ ]:
!nvidia-smi

## Step 2 — Install everything the backend needs

In [ ]:
!apt-get -qq install -y colmap ffmpeg imagemagick > /dev/null
!pip -q install fastapi uvicorn python-multipart pyngrok nest_asyncio plyfile tqdm opencv-python-headless
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting -q
%cd /content/gaussian-splatting
!pip -q install submodules/diff-gaussian-rasterization submodules/simple-knn
%cd /content
print('Setup complete.')

## Step 3 — Paste your ngrok authtoken
Replace `PASTE_YOUR_TOKEN_HERE` below with the token you copied from ngrok's site.

In [ ]:
NGROK_AUTHTOKEN = 'PASTE_YOUR_TOKEN_HERE'

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print('ngrok configured.')

## Step 4 — Bring in the backend code
This recreates `process_video.py` and `api.py` from the project's `/backend`
folder directly here in Colab, so this notebook is self-contained and
doesn't require uploading files manually.

In [ ]:
%%writefile /content/process_video.py
import argparse
import subprocess
import sys
from pathlib import Path


def run(cmd, **kwargs):
    print(f"\n$ {' '.join(cmd)}\n")
    result = subprocess.run(cmd, **kwargs)
    if result.returncode != 0:
        print(f"\n[ERROR] Command failed with exit code {result.returncode}: {' '.join(cmd)}")
        sys.exit(result.returncode)


def compress_video(video_path: Path, compressed_path: Path, max_width: int = 1280):
    """Step 0: shrink the video first. Smaller frames make COLMAP's
    matching step (the slowest, most CPU-heavy part) noticeably faster."""
    compressed_path.parent.mkdir(parents=True, exist_ok=True)
    run([
        'ffmpeg', '-y', '-i', str(video_path),
        '-vf', f"scale='min({max_width},iw)':-2",
        '-c:v', 'libx264', '-crf', '28', '-preset', 'fast',
        '-an',
        str(compressed_path),
    ])
    original_mb = video_path.stat().st_size / (1024 * 1024)
    compressed_mb = compressed_path.stat().st_size / (1024 * 1024)
    print(f'Compressed video: {original_mb:.1f} MB -> {compressed_mb:.1f} MB')


def extract_frames(video_path: Path, frames_dir: Path, fps: int = 2):
    frames_dir.mkdir(parents=True, exist_ok=True)
    run([
        'ffmpeg', '-y', '-i', str(video_path),
        '-qscale:v', '1', '-qmin', '1', '-vf', f'fps={fps}',
        str(frames_dir / 'frame_%04d.jpg'),
    ])
    frame_count = len(list(frames_dir.glob('*.jpg')))
    print(f'Extracted {frame_count} frames to {frames_dir}')
    if frame_count < 20:
        print('[WARNING] Fewer than 20 frames extracted. Consider a longer/slower walkthrough video.')


def run_colmap(project_dir: Path, frames_dir: Path):
    database_path = project_dir / 'database.db'
    sparse_dir = project_dir / 'sparse'
    sparse_dir.mkdir(parents=True, exist_ok=True)
    run(['colmap', 'feature_extractor', '--database_path', str(database_path), '--image_path', str(frames_dir)])
    run(['colmap', 'exhaustive_matcher', '--database_path', str(database_path)])
    run(['colmap', 'mapper', '--database_path', str(database_path), '--image_path', str(frames_dir), '--output_path', str(sparse_dir)])


def train_splat(project_dir: Path, output_dir: Path, iterations: int = 7000):
    run(['python3', '/content/gaussian-splatting/train.py', '-s', str(project_dir), '-m', str(output_dir), '--iterations', str(iterations)])


def export_ply(output_dir: Path, final_output: Path):
    candidates = sorted((output_dir / 'point_cloud').glob('iteration_*/point_cloud.ply'))
    if not candidates:
        print(f'[ERROR] No point_cloud.ply found under {output_dir}/point_cloud')
        sys.exit(1)
    final_output.parent.mkdir(parents=True, exist_ok=True)
    run(['cp', str(candidates[-1]), str(final_output)])
    print(f'\nDone. Final splat file: {final_output}')


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--fps', type=int, default=2)
    parser.add_argument('--iterations', type=int, default=7000)
    parser.add_argument('--max-width', type=int, default=1280, help='Downscale video to this max width first. Use 0 to skip.')
    args = parser.parse_args()

    video_path = Path(args.input)
    output_root = Path(args.output)
    compressed_path = output_root / 'compressed_input.mp4'
    project_dir = output_root / 'project'
    frames_dir = project_dir / 'images'
    training_dir = output_root / 'training'
    final_output = output_root / 'result.ply'

    if not video_path.exists():
        print(f'[ERROR] Input video not found: {video_path}')
        sys.exit(1)

    if args.max_width > 0:
        compress_video(video_path, compressed_path, max_width=args.max_width)
        video_to_process = compressed_path
    else:
        print('Skipping compression step (--max-width 0)')
        video_to_process = video_path

    extract_frames(video_to_process, frames_dir, fps=args.fps)
    run_colmap(project_dir, frames_dir)
    train_splat(project_dir, training_dir, iterations=args.iterations)
    export_ply(training_dir, final_output)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/api.py
import shutil
import subprocess
import uuid
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

app = FastAPI(title='RealMeta Video-to-3D Pipeline API (Colab)')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

BASE_DIR = Path('/content')
JOBS_DIR = BASE_DIR / 'jobs'
JOBS_DIR.mkdir(exist_ok=True)
JOBS = {}

# A free GPU can only realistically train one scene at a time. This flag
# makes the API refuse a new upload while one is already in progress,
# instead of letting several pile up and overwhelm the machine.
JOB_IN_PROGRESS = {'job_id': None}


def run_pipeline(job_id: str, video_path: Path, output_dir: Path):
    JOBS[job_id]['status'] = 'processing'
    try:
        subprocess.run(
            ['python3', str(BASE_DIR / 'process_video.py'), '--input', str(video_path), '--output', str(output_dir)],
            check=True,
        )
        JOBS[job_id]['status'] = 'done'
        JOBS[job_id]['result_path'] = str(output_dir / 'result.ply')
    except subprocess.CalledProcessError as e:
        JOBS[job_id]['status'] = 'failed'
        JOBS[job_id]['error'] = str(e)
    finally:
        if JOB_IN_PROGRESS['job_id'] == job_id:
            JOB_IN_PROGRESS['job_id'] = None


@app.post('/upload')
async def upload_video(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    if not file.filename.lower().endswith(('.mp4', '.mov')):
        raise HTTPException(status_code=400, detail='Please upload an .mp4 or .mov video file')
    if JOB_IN_PROGRESS['job_id'] is not None:
        raise HTTPException(
            status_code=429,
            detail='Another video is still processing. Please wait for it to finish before uploading another.',
        )
    job_id = str(uuid.uuid4())
    job_dir = JOBS_DIR / job_id
    job_dir.mkdir(parents=True)
    video_path = job_dir / file.filename
    with video_path.open('wb') as f:
        shutil.copyfileobj(file.file, f)
    JOBS[job_id] = {'status': 'pending', 'result_path': None, 'error': None}
    JOB_IN_PROGRESS['job_id'] = job_id
    background_tasks.add_task(run_pipeline, job_id, video_path, job_dir)
    return {'job_id': job_id, 'status': 'pending'}


@app.get('/status/{job_id}')
async def get_status(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail='Unknown job_id')
    return {'job_id': job_id, 'status': job['status'], 'error': job.get('error')}


@app.get('/result/{job_id}')
async def get_result(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail='Unknown job_id')
    if job['status'] != 'done':
        raise HTTPException(status_code=409, detail=f"Job is not finished yet (status: {job['status']})")
    return FileResponse(job['result_path'], filename='result.ply')

## Step 5 — Start the server and get your public URL

Run this cell and **leave it running** — it keeps the server alive.
Copy the printed URL and paste it into your website's "Backend URL" field.

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print('=' * 70)
print(f'Your backend is live at: {public_url}')
print('Paste this URL into the "Backend URL" field on your website.')
print('=' * 70)

import sys
sys.path.insert(0, '/content')
from api import app

# NOTE: we deliberately do NOT call uvicorn.run(app, ...) here. That
# convenience function calls asyncio.run() internally, which fails with
# "asyncio.run() cannot be called from a running event loop" on the
# newer Python versions Colab now uses, even with nest_asyncio applied.
# Building the Server object ourselves and driving it with
# run_until_complete() sidesteps that specific problem.
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

## To stop the server
Click the ⏹️ stop button on the cell above, or just close this tab. To
start again later, you'll need to re-run this notebook and paste the
**new** URL it gives you into your website.